## Educational Analytics Prediction Platform

### Data Cleaning

\Importing Necessary Libraries

In [32]:
import numpy as np
import pandas as pd
import os

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)


\Loading the dataset

In [33]:
df = pd.read_csv(r"C:\NG\Educational-Analytics-Platform\data\raw\student_dropout_dataset.csv")
target = "Dropout"

In [34]:
df.head()

,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,1,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,2,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,3,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,4,24.4,Male,NaN,Yes,NaN,82.2,2,38.6,No,No,NaN,1.78,1.77,1.77,Year 1,CS,High School,1
4,5,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0


\Missing Value Detection

In [35]:
#detecting date columns
def detect_datetime_columns(df):
    datetime_cols = []
    datetime_cols.extend(df.select_dtypes(include="datetime").columns.tolist())

    object_cols = df.select_dtypes(include="object").columns
    for col in object_cols:
        parsed = pd.to_datetime(df[col],errors= "coerce")
        success_ratio = parsed.notna().mean()
        if success_ratio >0.8:
            datetime_cols.append(col)
    return datetime_cols

In [36]:
def detect_missing_values(df):
    datetime_cols = detect_datetime_columns(df)
    numeric_cols = df.select_dtypes(include=np.number).columns
    categorical_cols = [c for c in df.select_dtypes(include="object").columns if c not in datetime_cols]

    missing_counts = df.isnull().sum()
    missing_counts = missing_counts[missing_counts>0]

    details = {}
    missing_numeric,missing_categorical,missing_datetime = [],[],[]

    for col ,count in missing_counts.items():
        if col in datetime_cols:
            col_type = "datetime"
            missing_datetime.append(col)
        elif col in numeric_cols:
            col_type = "numeric"
            missing_numeric.append(col) 
        elif col in categorical_cols:
            col_type = "categorical"
            missing_categorical.append(col) 

        details[col] = {
            "count": int(count),
            "pct" : round((count/len(df))*100,2),
            "type" :col_type,}

    return {
        "has_missing": len(missing_counts) > 0,
        "total_missing": int(missing_counts.sum()),
        "details": details,
        "numeric_cols": missing_numeric,
        "categorical_cols": missing_categorical,
        "datetime_cols": missing_datetime,
        "all_datetime_cols": datetime_cols,
    }
        

            

In [37]:
missing_report = detect_missing_values(df)
print("Has missing:", missing_report["has_missing"])
print("Total missing:", missing_report["total_missing"])
for col, d in missing_report["details"].items():
    print(f"  {col}: {d['count']} ({d['pct']}%) [{d['type']}]")

Has missing: True
Total missing: 2011
  Family_Income: 500 (5.0%) [numeric]
  Study_Hours_per_Day: 500 (5.0%) [numeric]
  Stress_Index: 500 (5.0%) [numeric]
  Parental_Education: 511 (5.11%) [categorical]


\Treating Missing Values

In [38]:
def treat_missing_values(df,missing_report, threshold=0.50,indicator_threshold=0.05):
    df_clean = df.copy()

    datetime_cols = missing_report["datetime_cols"]
    numeric_cols = missing_report["numeric_cols"]
    categorical_cols = missing_report["categorical_cols"]

    fill_values = {}
    dropped_cols = []
    filled_log = []
    indicator_cols = []

    for col, info in missing_report["details"].items():
        missing_count = info["count"]
        missing_ratio = missing_count/len(df_clean)

        if missing_ratio > threshold:
            df_clean = df_clean.drop(columns=[col])
            dropped_cols.append((col,round(missing_ratio *100,2)))
            continue

        if col in datetime_cols:
            df_clean[col] = pd.to_datetime(df_clean[col],errors="coerce")
            value = df_clean[col].median()
            df_clean[col]=df_clean[col].fillna(value)
            fill_values[col] = value
            filled_log.append((col,"median date",str(value.date()),missing_count))
            
        elif col in numeric_cols:
            value = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(value)
            fill_values[col] = value
            filled_log.append((col,"median",round(value,2),missing_count))

        elif col in categorical_cols:
            value = df_clean[col].mode()[0]
            df_clean[col] = df_clean[col].fillna(value)
            fill_values[col] = value
            filled_log.append((col, "mode", value, missing_count))

    report = {
            "dropped" : dropped_cols,
            "filled" : filled_log,
            "missing_after" : int(df_clean.isnull().sum().sum()),
    }
    return df_clean,fill_values,report

In [39]:
df, fill_values, treat_report = treat_missing_values(df, missing_report)
print("Dropped (>50%):", treat_report["dropped"] or "None")
for col, method, value, count in treat_report["filled"]:
    print(f"  {col}: filled {count} with {method} = {value}")
print("Missing after:", treat_report["missing_after"])

Dropped (>50%): None
  Family_Income: filled 500 with median = 29740.5
  Study_Hours_per_Day: filled 500 with median = 4.0
  Stress_Index: filled 500 with median = 5.5
  Parental_Education: filled 511 with mode = Bachelor
Missing after: 0


\Detect Duplicate Records

In [40]:
def detect_duplicates(df):
    full_dup = int(df.duplicated().sum())
    id_like = [col for col in df.columns if df[col].nunique() == len(df)]
    return {
        "full_duplicate_count": full_dup,
        "full_duplicate_pct": round((full_dup/len(df))*100,2),
    "id_like_columns": id_like,
    }

In [41]:
dup_report = detect_duplicates(df)
print("Full duplicates:", dup_report["full_duplicate_count"],
      f"({dup_report['full_duplicate_pct']}%)")
print("ID-like columns:", dup_report["id_like_columns"])

Full duplicates: 0 (0.0%)
ID-like columns: ['Student_ID']


\Removing Duplicate records

In [42]:
def remove_duplicates(df):
    rows_before = len(df)
    df_clean = df.drop_duplicates().reset_index(drop=True)
    return df_clean,rows_before - len(df_clean)
    

In [43]:
df_clean, rows_removed =remove_duplicates(df)
print("Rows removed:", rows_removed)
print("Shape now:", df.shape)

Rows removed: 0
Shape now: (10000, 19)


\Outlier Detection

In [44]:
def detect_outliers(df,target=None,min_unique=10):
    numeric_cols = df.select_dtypes(include = np.number).columns
    report = {}

    for col in numeric_cols:
        if target is not None and col==target:
            continue
        if df[col].nunique() < min_unique:
            continue

        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            continue

        lower = q1 -1.5 *iqr
        upper = q3 +1.5 *iqr
        outlier_mask = (df[col] < lower) | (df[col] > upper)
        count = int(outlier_mask.sum())

        if count > 0:
            report[col] = {
                "outlier_count": count,
                "outlier_pct" : round((count/len(df)) *100,2),
                "lower_bound": round(lower,2),
                "upper_bound": round(upper,2),
            }
            
    return report
        

In [45]:
outlier_report = detect_outliers(df, target=target)
print("Outliers detected in continuous columns:")
for col, info in outlier_report.items():
    print(f"  {col}: {info['outlier_count']} ({info['outlier_pct']}%) "
          f"bounds=[{info['lower_bound']}, {info['upper_bound']}]")

Outliers detected in continuous columns:
  Age: 33 (0.33%) bounds=[15.0, 27.0]
  Family_Income: 683 (6.83%) bounds=[-2542.62, 70904.38]
  Study_Hours_per_Day: 138 (1.38%) bounds=[0.81, 7.21]
  Attendance_Rate: 57 (0.57%) bounds=[60.05, 103.65]
  Travel_Time_Minutes: 25 (0.25%) bounds=[-2.85, 63.15]
  Stress_Index: 137 (1.37%) bounds=[1.1, 9.9]


In [46]:
def treat_outlier(df,outlier_report):
    df_clean = df.copy()
    capped_log = {}
    for col,info in outlier_report.items():
        df_clean[col] = df_clean[col].clip(lower=info["lower_bound"],upper=info["upper_bound"])
        capped_log[col] = info["outlier_count"]
    return df_clean,capped_log

In [47]:
df, capped_log = treat_outlier(df, outlier_report)
print("Total values capped:", sum(capped_log.values()))
print("Final shape:", df.shape)

Total values capped: 1073
Final shape: (10000, 19)


In [48]:
#master data clean fn
def clean_dataset(df, target =None):
    report = {"original_shape": df.shape}

    #duplicate 
    df,rows_removed = remove_duplicates(df)
    report["duplicates"] = {"rows_removed": rows_removed}

    #missing values
    missing_report = detect_missing_values(df)
    df, fill_values, treat_report = treat_missing_values(df,missing_report)
    report["missing"] = treat_report
    report["fill_values"] = fill_values

    #outlier
    outlier_report = detect_outliers(df, target=target)
    df, capped_log = treat_outlier(df, outlier_report)
    report["outliers"] = outlier_report

    report["final_shape"] = df.shape
    return df, report

In [49]:
def print_cleaning_report(report):
    print("=" * 55)
    print("  CLEANING REPORT")
    print("=" * 55)
    print(f"Original shape : {report['original_shape']}")
    print(f"Final shape    : {report['final_shape']}")

    # Missing
    m = report["missing"]
    print(f"\n[Missing Values]")
    print(f"  Dropped columns : {m['dropped'] or 'None'}")
    print(f"  Filled columns  : {len(m['filled'])}")
    for col, method, value, count in m["filled"]:
        print(f"     - {col} ({method} = {value}, {count} filled)")

    # Duplicates
    d = report["duplicates"]
    print(f"\n[Duplicates]")
    print(f"  Removed : {d['rows_removed']} rows")

    # Outliers
    o = report["outliers"]
    print(f"\n[Outliers] (capped in {len(o)} columns)")
    for col, info in o.items():
        print(f"     - {col}: {info['outlier_count']} capped")

In [50]:
df_raw = pd.read_csv(r"C:\NG\Educational-Analytics-Platform\data\raw\student_dropout_dataset.csv")
df_cleaned, full_report = clean_dataset(df_raw, target="Dropout")
print_cleaning_report(full_report)

  CLEANING REPORT
Original shape : (10000, 19)
Final shape    : (10000, 19)

[Missing Values]
  Dropped columns : None
  Filled columns  : 4
     - Family_Income (median = 29740.5, 500 filled)
     - Study_Hours_per_Day (median = 4.0, 500 filled)
     - Stress_Index (median = 5.5, 500 filled)
     - Parental_Education (mode = Bachelor, 511 filled)

[Duplicates]
  Removed : 0 rows

[Outliers] (capped in 6 columns)
     - Age: 33 capped
     - Family_Income: 683 capped
     - Study_Hours_per_Day: 138 capped
     - Attendance_Rate: 57 capped
     - Travel_Time_Minutes: 25 capped
     - Stress_Index: 137 capped
